In [ ]:
import pandas
import os

In [ ]:
samples = ['MM_339','MM_340','MM_341','MM_342','MM_343', 'MM_344', 'MM_345','MM_365','MM_366','MM_367', 'MM_371', 'MM_383', 'MM_384','MM_385','MM_386','MM_387','MM_388','MM_390', 'MM_391','MM_392','MM_394','MM_395','MM_396','MM_460','MM_535','MM_536','MM_537','MM_543','MM_544','MM_545','MM_546','MM_547','MM_548']

In [ ]:
for sample in samples:
    #reading in the single_cell.csv file from cellranger atac outputs
    wd = '/cellranger_outputs/'
    metrics = pandas.read_csv(os.path.join(wd, '{}/outs/singlecell.csv'.format(sample)), sep=',')
    metrics.head() 
    
    #reading in barcodes from our own "is cell" calls
    #reading in the list of good barcodes from snATAC individual QC 
    wd2 = '/data/'
    keep = pandas.read_csv(os.path.join(wd2, '{}_clean_min1k.txt'.format(sample)),sep='\t',header = 0,index_col=0)
    keep.head()
    
    keep.index = keep.index.str[7:]
    keep.index = keep.index.astype(str) +  '-1' 
    keep.head()
    
    len(keep)
    
    #creating and setting new is_cell column
    ind = 0
    new_is_cell = []
    for i in metrics['barcode']:
        if i in keep.index.values:
            new_is_cell.append(1)
        elif i not in keep.index.values:
            new_is_cell.append(0)
        if ind%100000 == 0:
            print(ind)
        ind = ind + 1
    
    metrics['is_cell'] = new_is_cell
    metrics.is_cell.value_counts()
    
    #replacing cellid column
    numid = 1
    ind = 1
    cellid = []
    for i in metrics['is_cell']:
        if i == 0:
            cellid = cellid + ["0"]
    
        elif i == 1:
            cellid = cellid + ["1"]
            numid = numid + 1
    
        ind = ind + 1
        if (ind%100000 == 0):
            print(ind)

    metrics['is__cell_barcode'] = cellid
    
    #save output
    wd3 = '/data/singlecell_files'
    metrics.to_csv(os.path.join(wd3, '{}_singlecell.csv'.format(sample)),index=False)
    print(sample)